# parameter-subclass-of-tensor — worked example 1: Parameter subclasses MiniTensor, default requires_grad

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-subclass-of-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `Parameter` IS-A `MiniTensor`: it subclasses the tensor type and only changes the default of `requires_grad` to `True`. Because it inherits, every `isinstance(_, MiniTensor)` gate in the autograd layer still accepts it, while the `Parameter` type tag marks it as trainable state.

## Worked solution

We define a minimal `MiniTensor` holding an array and a `requires_grad` flag, then a `Parameter(MiniTensor)` whose `__init__` simply forwards to `super().__init__` with `requires_grad=True` as the default. The subclass relationship is load-bearing: `isinstance(p, MiniTensor)` is `True`, so helpers that filter by `MiniTensor` see the Parameter. We construct a Parameter with the default and one with an explicit override, then print both `requires_grad` values and the two isinstance checks, confirming default-True, override works, and IS-A holds.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

p = Parameter([1.0, 2.0])
q = Parameter([3.0], requires_grad=False)
print('default rg:', p.requires_grad)
print('override rg:', q.requires_grad)
print('is MiniTensor:', isinstance(p, MiniTensor))